# Alternative Sentiment Scores

## Process the Data

- **Purpose:** Apply FinBERT to each ordered headline-and-summary text and retain its sentiment probabilities.
- **Settings:** `batch_size=16`; input text combines the headline and summary in news-ID order.
- **Data:** The ordered news text and IDs produce the stored sentiment-probability artifact.
- **Decision:** Reuse cached scores only when their ordered news IDs exactly match the current input.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.preprocessing.alternative_sentiment_scores import (
    score_sentiment_features,
)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
data_root = PROJECT_ROOT / "data/research_data/alternative"
period = "2025-01-01_2025-12-31"
news_path = data_root / "data" / f"aapl_{period}.parquet"
sentiment_path = data_root / "features" / f"aapl_finbert_sentiment_scores_{period}.parquet"
alternative_data = pd.read_parquet(news_path)
score_columns = [
    "sentiment_positive", "sentiment_negative",
    "sentiment_neutral", "sentiment_score",
]
cache_matches_input = False
if sentiment_path.is_file():
    sentiment_features = pd.read_parquet(sentiment_path)
    cache_matches_input = (
        "id" in sentiment_features
        and sentiment_features["id"].tolist() == alternative_data["id"].tolist()
    )
if not cache_matches_input:
    sentiment_features = score_sentiment_features(
        alternative_data,
        batch_size=16,
    )
    sentiment_path.parent.mkdir(parents=True, exist_ok=True)
    sentiment_features.to_parquet(sentiment_path, index=False)
sentiment_path

PosixPath('/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/alternative/features/aapl_finbert_sentiment_scores_2025-01-01_2025-12-31.parquet')

## Take a Quick Look at the Data Structure

- **Purpose:** Inspect the retained schema, source coverage, and positive, negative, neutral, and signed score distributions.
- **Settings:** No analytical parameters; read-only inspection.
- **Data:** Inspect the stored sentiment probabilities and their news-row alignment.
- **Decision:** Leave the stored values and row alignment unchanged.

In [2]:
sentiment_features.head()

,id,headline,source,url,summary,created_at,updated_at,symbols,author,content,sentiment_positive,sentiment_negative,sentiment_neutral,sentiment_score
0,42767369,'CPCS Secures Agreement With Apple For Enhance...,benzinga,https://www.benzinga.com/news/25/01/42767369/c...,,2025-01-02 15:32:15+00:00,2025-01-02 15:32:15+00:00,AAPL,Benzinga Newsdesk,,0.928484,0.013745,0.057771,0.914739
1,42773823,Some Investors Are Selling Apple Stock Thursda...,benzinga,https://www.benzinga.com/news/global/25/01/427...,Apple Inc (NASDAQ:AAPL) shares are trading low...,2025-01-02 19:25:09+00:00,2025-01-02 19:25:10+00:00,AAPL,Adam Eckert,<p><strong>Apple Inc</strong> (NASDAQ:<a class...,0.009012,0.972913,0.018075,-0.963900
2,42780075,Apple Settles $95 Million Lawsuit Over Siri Pr...,benzinga,https://www.benzinga.com/news/global/25/01/427...,Apple Inc. has agreed to pay $95 million to se...,2025-01-03 03:59:46+00:00,2025-01-03 03:59:47+00:00,AAPL,Kaustubh Bagalkote,<p><strong>Apple Inc. </strong>(NASDAQ:<a clas...,0.063167,0.444193,0.492640,-0.381026
3,42784735,"B of A Securities Maintains Buy on Apple, Main...",benzinga,https://www.benzinga.com/news/25/01/42784735/b...,,2025-01-03 13:57:22+00:00,2025-01-03 13:57:22+00:00,AAPL,Benzinga Newsdesk,,0.035534,0.019706,0.944760,0.015828
4,42788545,"Bernstein Maintains Outperform on Apple, Raise...",benzinga,https://www.benzinga.com/news/25/01/42788545/b...,,2025-01-03 15:30:38+00:00,2025-01-03 15:30:39+00:00,AAPL,Benzinga Newsdesk,,0.888683,0.046740,0.064577,0.841944


In [3]:
sentiment_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   id                  395 non-null    int64              
 1   headline            395 non-null    str                
 2   source              395 non-null    str                
 3   url                 395 non-null    str                
 4   summary             395 non-null    str                
 5   created_at          395 non-null    datetime64[us, UTC]
 6   updated_at          395 non-null    datetime64[us, UTC]
 7   symbols             395 non-null    str                
 8   author              395 non-null    str                
 9   content             395 non-null    str                
 10  sentiment_positive  395 non-null    float64            
 11  sentiment_negative  395 non-null    float64            
 12  sentiment_neutral   395 non-null    float64    

In [4]:
sentiment_features["source"].value_counts(dropna=False)

source
benzinga    395
Name: count, dtype: int64

In [5]:
sentiment_features[score_columns].describe()

,sentiment_positive,sentiment_negative,sentiment_neutral,sentiment_score
count,395.000000,395.000000,395.000000,395.000000
mean,0.332275,0.229344,0.438381,0.102931
std,0.344205,0.331162,0.351216,0.577008
min,0.007474,0.006566,0.012556,-0.964634
25%,0.039745,0.016904,0.082508,-0.275225
50%,0.156018,0.033919,0.368746,0.087597
75%,0.662145,0.411167,0.828100,0.597449
max,0.957127,0.973652,0.949904,0.941092


In [6]:
sentiment_features[score_columns].hist(bins=30, figsize=(10, 8))
plt.tight_layout()
plt.show()

/var/folders/1z/bcvql7210c77v6rjkswpzsyr0000gn/T/ipykernel_61189/2085060402.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
